# Is this Steam game worth playing?

This notebook trains a small model that answers one question: **"Will players like this game?"**

You describe a game with simple facts (free or paid, price, genres, platforms ...) and the model gives you
**Worth playing / Not worth playing** plus a probability. Because it uses facts and not the game's *name*,
it also works for **games that are not in the dataset** (new, upcoming or made-up games).

**What "worth playing" means:** at least **80% positive reviews** (Steam's "Very Positive" line) **and** at least **50 reviews**.

**Data (add both on Kaggle with *Add Input*):**
1. `extended_steam_games.csv` - game details (genres, price, languages ...)
2. *Game Recommendations on Steam* (`antonkozyriev/game-recommendations-on-steam`) - only its `games.csv`, used for the review score (to create the answer)

**No cheating:** review scores are only used to create the answer. They are never used as model inputs.

**Steps:** 1 Load data - 2 Make the answer column - 3 Make the input columns - 4 Charts - 5 Train and compare models -
6 Test - 7 Predict for a new game - 8 Save for your UI - 9 Try it yourself

In [ ]:
import os
import json
from datetime import date
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
MIN_REVIEWS = 50     # a game needs at least this many reviews to get an answer
GOOD_RATIO = 80      # % positive reviews needed to count as "worth playing"

## 1. Load the data

In [ ]:
ROOT = Path(os.environ.get("STEAM_INPUT", "/kaggle/input"))

# Find the two files automatically (the second dataset's games.csv is the one with an "app_id" column)
details_path = next(ROOT.rglob("extended_steam_games.csv"), None)
reviews_path = next((p for p in ROOT.rglob("games.csv") if "app_id" in pd.read_csv(p, nrows=0).columns), None)

if details_path is None or reviews_path is None:
    raise FileNotFoundError("Attach BOTH datasets with 'Add Input' (extended_steam_games.csv and "
                            "antonkozyriev/game-recommendations-on-steam).")

details = pd.read_csv(details_path, low_memory=False).drop_duplicates("appid")
reviews = pd.read_csv(reviews_path, usecols=["app_id", "positive_ratio", "user_reviews"]).drop_duplicates("app_id")

# Join the two tables using the Steam game id
data = details.merge(reviews, left_on="appid", right_on="app_id", how="inner")
print("Games in both datasets:", len(data))

## 2. Make the answer column (worth playing = 1, not worth playing = 0)

In [ ]:
if data["positive_ratio"].max() <= 1:                 # in case the ratio is stored as 0-1 instead of 0-100
    data["positive_ratio"] = data["positive_ratio"] * 100

data = data[data["user_reviews"] >= MIN_REVIEWS].copy()          # ignore games with too few reviews
data["worth_playing"] = (data["positive_ratio"] >= GOOD_RATIO).astype(int)

print("Games with enough reviews:", len(data))
print(data["worth_playing"].value_counts().rename({1: "worth playing", 0: "not worth playing"}))

## 3. Make the input columns
These are the **only things the model will know about a game**. They are also exactly what your UI will ask the user for.

| Input | Meaning |
|---|---|
| `is_free`, `price_usd` | free or paid, and the price |
| `windows`, `mac`, `linux` | platforms (1 = yes, 0 = no) |
| `n_languages`, `required_age`, `achievements`, `release_year` | simple numbers |
| `genres`, `categories` | text lists like `"Action,Indie"` and `"Single-player,Steam Cloud"` |

In [ ]:
def yes_no(column):                       # "True"/"False" text -> 1/0
    return data[column].astype(str).str.lower().eq("true").astype(int)

def count_items(text):                    # "English, French" -> 2
    return text.fillna("").map(lambda s: len([x for x in s.split(",") if x.strip()]))

def clean_list(text):                     # "Action, Indie" -> "Action,Indie"
    return text.fillna("").str.replace(r"\s*,\s*", ",", regex=True).str.strip()

languages = data["supported_languages"].fillna("").str.split("<br>").str[0]    # drop the html footnote
release_date = pd.to_datetime(data["release_date"], format="mixed", errors="coerce")

X = pd.DataFrame({
    "is_free":      yes_no("is_free"),
    "price_usd":    pd.to_numeric(data["initialprice"], errors="coerce") / 100,   # prices are stored in cents
    "windows":      yes_no("platforms_windows"),
    "mac":          yes_no("platforms_mac"),
    "linux":        yes_no("platforms_linux"),
    "n_languages":  count_items(languages),
    "required_age": pd.to_numeric(data["required_age"], errors="coerce"),
    "achievements": pd.to_numeric(data["achievements"], errors="coerce"),
    "release_year": release_date.dt.year,
    "genres":       clean_list(data["genres"]),
    "categories":   clean_list(data["categories"]),
})
y = data["worth_playing"]

numeric_cols = ["is_free", "price_usd", "windows", "mac", "linux",
                "n_languages", "required_age", "achievements", "release_year"]
X.head()

## 4. A few simple charts

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

counts = y.value_counts().sort_index()
ax[0].bar(["Not worth playing", "Worth playing"], counts.values, color=["#d9534f", "#5cb85c"])
ax[0].set_title("Games in each group"); ax[0].set_ylabel("Games")

ax[1].hist(data["positive_ratio"], bins=30, color="#4a90d9", edgecolor="white")
ax[1].axvline(GOOD_RATIO, color="red", linestyle="--", label=f"{GOOD_RATIO}% = worth playing")
ax[1].set_title("% positive reviews per game"); ax[1].set_xlabel("% positive"); ax[1].legend()

rate = (y.groupby(X["is_free"]).mean() * 100).rename({0: "Paid", 1: "Free"})
ax[2].bar(rate.index, rate.values, color=["#4a90d9", "#f0ad4e"][:len(rate)])
ax[2].set_title("% worth playing: paid vs free"); ax[2].set_ylabel("% worth playing")

plt.tight_layout(); plt.show()

## 5. Train and compare models
We keep 20% of the games aside as a **test set** that we only look at once, at the end.
Everything the model needs (filling blanks, scaling numbers, turning genres into yes/no columns) is inside one **pipeline**,
so a UI can send in a plain game description and the pipeline does the rest.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

# The genres and categories the model knows about = the most common ones in the training games.
# Your UI will show exactly these as choices.
def most_common(text, n):
    items = text.str.split(",").explode()
    return sorted(items[items != ""].value_counts().head(n).index)      # n most common, A-Z

GENRE_OPTIONS = most_common(X_train["genres"], 20)
CATEGORY_OPTIONS = most_common(X_train["categories"], 12)
print("Genres the model knows:    ", GENRE_OPTIONS)
print("Categories the model knows:", CATEGORY_OPTIONS)

In [ ]:
def make_pipeline(model):
    prepare = ColumnTransformer([
        # numbers: fill blanks with the median, then put them on a similar scale
        ("numbers", Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
        # genres / categories: one yes/no column per option
        ("genres", CountVectorizer(vocabulary=GENRE_OPTIONS, token_pattern=r"[^,]+", lowercase=False, binary=True), "genres"),
        ("categories", CountVectorizer(vocabulary=CATEGORY_OPTIONS, token_pattern=r"[^,]+", lowercase=False, binary=True), "categories"),
    ], sparse_threshold=0)
    return Pipeline([("prepare", prepare), ("model", model)])

models = {
    "Baseline (always guess the majority)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest":       RandomForestClassifier(n_estimators=150, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting":   HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE),
}

# 5-fold cross-validation on the training set (F1 = balance of precision and recall, higher is better)
cv_f1 = {}
for name, model in models.items():
    scores = cross_val_score(make_pipeline(model), X_train, y_train, cv=5, scoring="f1")
    cv_f1[name] = scores.mean()
    print(f"{name:38s} F1 = {scores.mean():.3f}")

plt.barh(list(cv_f1.keys()), list(cv_f1.values()), color="#4a90d9")
plt.title("Cross-validated F1 score (higher is better)"); plt.gca().invert_yaxis(); plt.show()

In [ ]:
# Pick the best real model (not the baseline) and train it on all the training data
best_name = max((n for n in cv_f1 if not n.startswith("Baseline")), key=cv_f1.get)
print("Best model:", best_name)

final_model = make_pipeline(models[best_name]).fit(X_train, y_train)

## 6. Test it on games it has never seen

In [ ]:
pred = final_model.predict(X_test)
proba = final_model.predict_proba(X_test)[:, 1]

results = pd.Series({"Accuracy": accuracy_score(y_test, pred), "Precision": precision_score(y_test, pred),
                     "Recall": recall_score(y_test, pred), "F1": f1_score(y_test, pred),
                     "ROC-AUC": roc_auc_score(y_test, proba)})
print(results.round(3).to_string())
print("\nAccuracy of always guessing the majority:", round(max(y_test.mean(), 1 - y_test.mean()), 3))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["Not worth", "Worth"], cmap="Blues", ax=ax[0], colorbar=False)
ax[0].set_title("Right and wrong guesses"); ax[0].grid(False)

RocCurveDisplay.from_predictions(y_test, proba, ax=ax[1])
ax[1].plot([0, 1], [0, 1], "k--", label="Random guess"); ax[1].set_title("ROC curve"); ax[1].legend()

# Which inputs matter most? Shuffle one input at a time and see how much the score drops.
imp = permutation_importance(final_model, X_test, y_test, scoring="f1", n_repeats=5, random_state=RANDOM_STATE)
imp = pd.Series(imp.importances_mean, index=X_test.columns).sort_values().tail(10)
ax[2].barh(imp.index, imp.values, color="#5cb85c"); ax[2].set_title("Most useful inputs"); ax[2].set_xlabel("Drop in F1 when shuffled")

plt.tight_layout(); plt.show()

## 7. Predict for ANY game by describing it
No game name needed. Give the facts you know and `predict_game` returns the verdict.
The **probability** is the model's chance that the game reaches 80%+ positive reviews.

> This function is also what your UI will call. It only needs `pandas` and the trained model.

In [ ]:
def predict_game(model, is_free, price_usd=0.0, genres=(), categories=(),
                 windows=True, mac=False, linux=False,
                 n_languages=1, required_age=0, achievements=0, release_year=None):
    """Describe a game -> get a verdict. genres / categories are lists of text, e.g. ["Action", "Indie"]."""
    if is_free:
        price_usd = 0.0
    game = pd.DataFrame([{
        "is_free":      int(is_free),
        "price_usd":    float(price_usd),
        "windows":      int(windows),
        "mac":          int(mac),
        "linux":        int(linux),
        "n_languages":  n_languages,
        "required_age": required_age,
        "achievements": achievements,
        "release_year": release_year or date.today().year,
        "genres":       ",".join(genres),          # ["Action", "Indie"] -> "Action,Indie"
        "categories":   ",".join(categories),
    }])
    probability = float(model.predict_proba(game)[0, 1])
    worth = probability >= 0.5
    return {"worth_playing": worth,
            "probability": round(probability, 3),
            "verdict": "Worth playing" if worth else "Not worth playing"}

In [ ]:
# Two example games (change the values and re-run!)
examples = {
    "Free multiplayer shooter": dict(is_free=True, genres=["Action", "Free To Play"], categories=["Multi-player", "PvP"],
                                     windows=True, n_languages=10, achievements=0),
    "Paid single-player RPG":   dict(is_free=False, price_usd=29.99, genres=["RPG", "Adventure"],
                                     categories=["Single-player", "Steam Achievements", "Full controller support"],
                                     windows=True, mac=True, n_languages=8, achievements=40),
}
for title, game in examples.items():
    print(title, "->", predict_game(final_model, **game))

## 8. Save the model for your UI
We save **one file** with the trained pipeline *and* the lists of genres / categories, so the UI knows what to show in its dropdowns.
A small JSON file with the same lists is saved too (handy for non-Python front ends).

In [ ]:
out = Path(os.environ.get("STEAM_OUT", "/kaggle/working")); out.mkdir(parents=True, exist_ok=True)

bundle = {
    "model": final_model,                     # the full pipeline: send in a game description, get a probability
    "genre_options": GENRE_OPTIONS,           # choices for the "genres" selector
    "category_options": CATEGORY_OPTIONS,     # choices for the "game features" selector
    "good_ratio": GOOD_RATIO,
    "min_reviews": MIN_REVIEWS,
}
joblib.dump(bundle, out / "worth_playing_model.joblib")
(out / "ui_options.json").write_text(json.dumps({"genres": GENRE_OPTIONS, "categories": CATEGORY_OPTIONS}, indent=2))
print("Saved:", out / "worth_playing_model.joblib", "and", out / "ui_options.json")

# Check that the saved file works on its own (this is what your UI will do)
loaded = joblib.load(out / "worth_playing_model.joblib")
print(predict_game(loaded["model"], is_free=False, price_usd=14.99, genres=["Indie", "Action"], categories=["Single-player"]))

### How to use the saved model in your UI
Your UI needs to collect these inputs, then call the same `predict_game` function (copy it from section 7 into your app file):

| UI control | Argument | Choices |
|---|---|---|
| Toggle: Free / Paid | `is_free` | `True` / `False` |
| Number box (hide if free) | `price_usd` | e.g. 0 - 100 |
| Multi-select | `genres` | `bundle["genre_options"]` |
| Multi-select | `categories` | `bundle["category_options"]` |
| 3 checkboxes | `windows`, `mac`, `linux` | `True` / `False` |
| Number boxes | `n_languages`, `required_age`, `achievements`, `release_year` | whole numbers |

```python
import joblib
bundle = joblib.load("worth_playing_model.joblib")
result = predict_game(bundle["model"], is_free=False, price_usd=19.99,
                      genres=["Indie", "Action"], categories=["Single-player"])
# result -> {"worth_playing": True/False, "probability": 0.0-1.0, "verdict": "..."}
```

Use the **same scikit-learn version** in your UI as in this notebook, otherwise loading the file can fail.

## 9. Try it yourself - answer a few questions
Run the next two cells. It asks about the game (free or paid, price, genres, platforms ...) and then gives the prediction.

In [ ]:
def ask_yes_no(question, default="n"):
    answer = input(f"{question} (y/n) [{default}]: ").strip().lower() or default
    return answer.startswith("y")

def ask_number(question, default):
    text = input(f"{question} [{default}]: ").strip()
    try:
        return float(text) if text else default
    except ValueError:
        print("  That is not a number, using", default)
        return default

def ask_choices(question, options):
    print(f"\n{question}")
    for i, name in enumerate(options, 1):
        print(f"  {i}. {name}")
    text = input("Type the numbers separated by commas, e.g. 1,3 (or press Enter to skip): ")
    chosen = []
    for part in text.split(","):
        part = part.strip()
        if part.isdigit() and 1 <= int(part) <= len(options):
            chosen.append(options[int(part) - 1])
    return chosen

def ask_and_predict(model):
    print("Describe the game:\n")
    is_free = ask_yes_no("Is the game free?")
    price = 0.0 if is_free else ask_number("Price in USD", 19.99)
    genres = ask_choices("Genres", GENRE_OPTIONS)
    categories = ask_choices("Game features", CATEGORY_OPTIONS)
    print("\nPlatforms:")
    windows = ask_yes_no("  Windows?", "y")
    mac = ask_yes_no("  Mac?")
    linux = ask_yes_no("  Linux?")
    print()
    n_languages = int(ask_number("Number of supported languages", 1))
    age = int(ask_number("Required age (0 = everyone)", 0))
    achievements = int(ask_number("Number of achievements", 0))
    year = int(ask_number("Release year", date.today().year))

    result = predict_game(model, is_free, price, genres, categories, windows, mac, linux,
                          n_languages, age, achievements, year)
    print(f"\n==> {result['verdict'].upper()}  (chance of 80%+ positive reviews: {result['probability']:.0%})")

In [ ]:
ask_and_predict(final_model)       # run this cell as many times as you like

## Notes and limits
- The answer comes from Steam players' reviews, so the model learns **what kinds of games tend to get good reviews**, not whether *you* will like a game.
- It only sees basic facts. It cannot know how good the gameplay, art or marketing is, so treat the result as a hint, not a verdict.
- Genres or features that are not in the lists above are ignored by the model.
- Games with fewer than 50 reviews are left out of training, so brand-new or very obscure games are the hardest to judge.
- Change `GOOD_RATIO` or `MIN_REVIEWS` at the top and re-run everything to see the effect.